# <center> Deliverable 02

### Task 1 - File Ingestion and reguar expressions

In [ ]:
from pathlib import Path

import re

import pandas as pd

# Task 1.1 
def read_log_file(file_path):
    path = Path(file_path)

    try:
        content = path.read_text(encoding="utf-8")
        return content, "utf-8"

    except UnicodeDecodeError:
        content = path.read_text(encoding="iso-8859-1")
        return content, "iso-8859-1"



# Task 1.2

def log_parser(log):
    log_lines = log.split('\n')
    logs = []
    pattern = r"\[(\d{4}-\d{2}-\d{2} \d{2}:\d{2}:\d{2})\] \[(POS_\d+)\] item=(\w+) price=(\d+\.\d+) (\w+) \((\w+)\)"
    
    for log_line in log_lines:
        match = re.match(pattern, log_line)
        if match:
            log_dict = {
                "timestamp": match.group(1),
                "register_id": match.group(2),
                "item_name": match.group(3),
                "price": match.group(4),
                "currency": match.group(5),
                "status": match.group(6)
            }
            logs.append(log_dict)
    
    return logs


### Task 2 - Recursive utility functions

In [16]:
# Task 2.1

def find_log_files(directory):
    directory = Path(directory)
    log_files = []

    for item in directory.iterdir():

        if item.is_dir():
            log_files.extend(find_log_files(item))

        elif item.is_file() and item.suffix == ".log":
            log_files.append(item)

    return log_files


# Task 2.2

def find_active_registers(data) -> set[str]:
    active_registers = set()

    if isinstance(data, dict):

        if data.get("active") is True and isinstance(data.get("id"), str):
            active_registers.add(data["id"])

        for value in data.values():
            active_registers.update(find_active_registers(value))

    elif isinstance(data, list):

        for item in data:
            active_registers.update(find_active_registers(item))

    return active_registers



### Task 3 - Functions for cleaning and analysing data

### Task 4 - Data analysis using Pandas

In [ ]:
# Task 4.1 creating dataframe
def create_dataframe(transactions):
    df = pd.DataFrame(transactions)

    df["timestamp"] = pd.to_datetime(df["timestamp"])

    df = df.set_index("timestamp")

    df = df.sort_index()

    return df

# task 4.2 revenue per item per hour
def hourly_revenue(df):
    revenue = (
        df.groupby("item")
        .resample("1H")["price"]
        .sum()
        .reset_index()
    )

    return revenue

# Task 4.3 Pivot table 



### Task 5 - Create full workflow